In [5]:
import numpy as np
from pathlib import Path
import sys

for path in (Path.cwd(), Path.cwd() / "certificates" / "empirical_laws", Path.cwd().parent / "certificates" / "empirical_laws"):
    if (path / "notebook_setup.py").exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

import notebook_setup
from lyapunov import has_lyapunov
from utils import dask_parallel_map


EPSILONS = np.linspace(0.05, 0.95, 10)
L_VALUES = [1, 100, 1000.0]
KAPPA_VALUES = [2, 100, 1000]
N_WORKERS = [2, 3, 4]

TEST_IMPROVEMENT = 1 - 1e-2
SIMPLIFIED_DIRECT_CHECK_RELATIVE_MARGIN = 1e-4
SIMPLIFIED_BISECTION_TOL = 1e-7
SIMPLIFIED_LYAP_SOLVER = "MOSEK"
SIMPLIFIED_LYAP_SOLVE_KWARGS = notebook_setup.MOSEK_SINGLE_THREAD_SOLVE_KWARGS
SIMPLIFIED_DIRECT_CHECK_SOLVE_KWARGS = notebook_setup.MOSEK_STRICT_SOLVE_KWARGS
FULL_LYAP_SOLVER = "MOSEK"
FULL_LYAP_SOLVE_KWARGS = notebook_setup.MOSEK_STRICT_SOLVE_KWARGS
FULL_LYAP_CONFIRM_SOLVER = "SDPA"
FULL_LYAP_CONFIRM_SOLVE_KWARGS = dict(notebook_setup.SDPA_HIGH_PRECISION_SOLVE_KWARGS)
DASK_SCHEDULER = "processes"
DASK_NUM_WORKERS = 14


def base_lyapunov_kwargs(*, epsilon, n_workers, mus, Ls, use_simplified_lyapunov):
    return {
        "delta": 1 - epsilon,
        "n_workers": n_workers,
        "mus": mus,
        "Ls": Ls,
        "method": "EF21",
        "use_simplified_lyapunov": use_simplified_lyapunov,
        "homogenous": False,
        "log_det_iterations": 0,
    }


def check_config(args):
    L_tuple, kappa_tuple, epsilon, n_workers = args
    scale_info = notebook_setup.scaled_problem_data_for_case(L_tuple, kappa_tuple)
    Ls = scale_info["Ls"]
    mus = scale_info["mus"]
    scale_label = notebook_setup.scale_label(scale_info)
    eta_star = notebook_setup.empirical_eta_star(epsilon, Ls, mus, n_workers)

    simplified_kwargs = base_lyapunov_kwargs(
        epsilon=epsilon,
        n_workers=n_workers,
        mus=mus,
        Ls=Ls,
        use_simplified_lyapunov=True,
    )
    rho_simplified, _, _, warned_midpoints = notebook_setup.warning_aware_bisection(
        0.0,
        1.0,
        SIMPLIFIED_BISECTION_TOL,
        has_lyapunov,
        eta=float(eta_star),
        solver=SIMPLIFIED_LYAP_SOLVER,
        solve_kwargs=SIMPLIFIED_LYAP_SOLVE_KWARGS,
        **simplified_kwargs,
    )
    if rho_simplified is None:
        return (
            False,
            np.nan,
            epsilon,
            L_tuple,
            kappa_tuple,
            f"Could not certify simplified reference with {scale_label}: "
            f"simplified reference bisection failed (eta={eta_star:.8g}, warned_midpoints={warned_midpoints})",
        )

    full_kwargs = base_lyapunov_kwargs(
        epsilon=epsilon,
        n_workers=n_workers,
        mus=mus,
        Ls=Ls,
        use_simplified_lyapunov=False,
    )
    rho_imp = rho_simplified * TEST_IMPROVEMENT
    full_info = notebook_setup.warning_aware_solve(
        has_lyapunov,
        rho_imp,
        eta=float(eta_star),
        solver=FULL_LYAP_SOLVER,
        solve_kwargs=FULL_LYAP_SOLVE_KWARGS,
        **full_kwargs,
    )
    detail = f"{scale_label} eta={eta_star:.8g} rho_simplified={float(rho_simplified):.10f}"
    if full_info["ok_clean"]:
        simplified_direct_checks = []
        for label, rho_test in (
            ("rho_imp", rho_imp),
            ("rho_imp_strict", rho_imp * (1 - SIMPLIFIED_DIRECT_CHECK_RELATIVE_MARGIN)),
        ):
            check_info = notebook_setup.warning_aware_solve(
                has_lyapunov,
                rho_test,
                eta=float(eta_star),
                solver=SIMPLIFIED_LYAP_SOLVER,
                solve_kwargs=SIMPLIFIED_DIRECT_CHECK_SOLVE_KWARGS,
                **simplified_kwargs,
            )
            simplified_direct_checks.append((label, float(rho_test), check_info))

        simplified_direct_detail = " ".join(
            f"{label}={rho_test:.10f} {notebook_setup.solve_info_label(check_info)}"
            for label, rho_test, check_info in simplified_direct_checks
        )
        if any(check_info["ok_clean"] for _, _, check_info in simplified_direct_checks):
            return (
                True,
                float(rho_simplified),
                epsilon,
                L_tuple,
                kappa_tuple,
                "No full-class improvement beyond the simplified direct check: "
                + detail
                + " "
                + simplified_direct_detail,
            )

        full_confirm_info = notebook_setup.warning_aware_solve(
            has_lyapunov,
            rho_imp,
            eta=float(eta_star),
            solver=FULL_LYAP_CONFIRM_SOLVER,
            solve_kwargs=FULL_LYAP_CONFIRM_SOLVE_KWARGS,
            **full_kwargs,
        )
        confirm_detail = (
            f"{detail} mosek={notebook_setup.solve_info_label(full_info)} "
            f"sdpa={notebook_setup.solve_info_label(full_confirm_info)} "
            f"simplified_direct={simplified_direct_detail}"
        )
        if not full_confirm_info["ok_clean"]:
            return (
                True,
                float(rho_simplified),
                epsilon,
                L_tuple,
                kappa_tuple,
                "No SDPA-confirmed full-class improvement: " + confirm_detail,
            )
        return (
            False,
            float(rho_imp),
            epsilon,
            L_tuple,
            kappa_tuple,
            "Full improves on the clean max-L normalized case, confirmed by SDPA: " + confirm_detail,
        )

    return (
        True,
        float(rho_simplified),
        epsilon,
        L_tuple,
        kappa_tuple,
        "No clean full-class improvement: " + detail,
    )

def verify_lyapunov_rigorous(n_workers=2):
    worker_configs = notebook_setup.worker_L_kappa_configs(
        L_VALUES,
        KAPPA_VALUES,
        n_workers,
        dedup_permutations=True,
    )
    configs = [
        (L_cfg, kappa_cfg, float(epsilon), n_workers)
        for (L_cfg, kappa_cfg) in worker_configs
        for epsilon in EPSILONS
    ]
    print(f"--- Verification Suite: Empirical Law 4.2 (Lyapunov), n={n_workers} ---")
    print(f"Checking {len(configs)} configurations with {DASK_NUM_WORKERS} workers...")
    if not configs:
        print("No configurations in this check.\n")
        return []

    results = dask_parallel_map(
        check_config,
        configs,
        scheduler=DASK_SCHEDULER,
        num_workers=DASK_NUM_WORKERS,
    )
    failures = []
    for result in results:
        passed, err, eps, L_cfg, kappa_cfg, msg = result
        if not passed:
            failures.append(result)
            print(f"FAIL: L={L_cfg}, kappa={kappa_cfg}, Eps={eps:.3f} | {msg} (Val={err})")

    if not failures:
        print("ALL CHECKS PASSED\n")
    else:
        print(f"FAILURES={len(failures)}\n")
    return failures


if __name__ == "__main__":
    total_failures = []
    for n_workers in N_WORKERS:
        total_failures.extend(verify_lyapunov_rigorous(n_workers=n_workers))

    if total_failures:
        print(f"TOTAL_FAILURES={len(total_failures)}")
        raise SystemExit(1)
    print("ALL LYAPUNOV CHECKS PASSED")

--- Verification Suite: Empirical Law 4.2 (Lyapunov), n=2 ---
Checking 450 configurations with 14 workers...


compute: 100%|██████████| 450/450 [00:31<00:00, 14.37it/s]


ALL CHECKS PASSED

--- Verification Suite: Empirical Law 4.2 (Lyapunov), n=3 ---
Checking 1650 configurations with 14 workers...


compute: 100%|██████████| 1650/1650 [04:22<00:00,  6.28it/s]


ALL CHECKS PASSED

--- Verification Suite: Empirical Law 4.2 (Lyapunov), n=4 ---
Checking 4950 configurations with 14 workers...


compute: 100%|██████████| 4950/4950 [15:42<00:00,  5.25it/s]


ALL CHECKS PASSED

ALL LYAPUNOV CHECKS PASSED
